Imlo coursework

In [1]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import torchvision
import torchvision.transforms as transforms

In [2]:
# code to use my gpu
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

print(device)

cuda


In [3]:
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

In [4]:
#defining the train and val
train_data = torchvision.datasets.OxfordIIITPet(
    root = './data',
    split = 'trainval',
    transform = transform,
    download = True
)

#splitting trainval
train_size = int(0.8 * len(train_data))
val_size = len(train_data) - train_size
train_data, val_data = torch.utils.data.random_split(
    train_data,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42) #makes sure that the images used in train and val stick tg
)

#dataloaders for train and val
train_loader = torch.utils.data.DataLoader(train_data, batch_size=32, shuffle=True, num_workers=2)
val_loader = torch.utils.data.DataLoader(val_data, batch_size=32, shuffle=True, num_workers=2)

100%|██████████| 792M/792M [00:44<00:00, 17.7MB/s]
100%|██████████| 19.2M/19.2M [00:01<00:00, 11.0MB/s]


In [5]:
image, label = train_data[0]

In [6]:
image.size()

torch.Size([3, 128, 128])

In [7]:
# Adding names for the catergories
class_names = train_data.dataset.classes

In [8]:
# Defining the layers
class NeuralNet(nn.Module):
    def __init__(self):
    #calls constructor from nn.Module
        super().__init__()

        self.conv1 = nn.Conv2d(3, 12, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(12, 24, 5)

        self.fc1 = nn.Linear(24 * 29 * 29, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 37)

    def forward(self, input):
        input = self.pool(F.relu(self.conv1(input)))  #applying conv1, then RELU, then pooling layer
        input = self.pool(F.relu(self.conv2(input)))  #applying conv2, then RELU then pooling layer
        input = torch.flatten(input, 1)  #flattening
        input = F.relu(self.fc1(input))  #applying fc1, then RELU
        input = F.relu(self.fc2(input))  #applying fc2, then RELU
        input = self.fc3(input)  #applying fc3
        return input

In [9]:
# defining the NN itself
network = NeuralNet().to(device)
loss_func = nn.CrossEntropyLoss()
optimiser = torch.optim.Adam(network.parameters(), lr=0.001)

In [10]:
# training the model
for epoch in range(30):
    print("Training epoch:", epoch)
    running_loss = 0.0

    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimiser.zero_grad()

        outputs = network(inputs)
        loss = loss_func(outputs, labels)
        loss.backward()
        optimiser.step()

        running_loss += loss.item()

    running_loss_calc = running_loss / len(train_loader)
    print("Loss:", running_loss_calc)

Training epoch: 0
Loss: 3.5845165589581365
Training epoch: 1
Loss: 3.466983525649361
Training epoch: 2
Loss: 3.337456892365995
Training epoch: 3
Loss: 3.0750181052995766
Training epoch: 4
Loss: 2.6368752303330796
Training epoch: 5
Loss: 1.9937606440938038
Training epoch: 6
Loss: 1.2827709796636
Training epoch: 7
Loss: 0.6449272357251333
Training epoch: 8
Loss: 0.25975383307946764
Training epoch: 9
Loss: 0.11798815700508979
Training epoch: 10
Loss: 0.06769852575076662
Training epoch: 11
Loss: 0.07199059249630765
Training epoch: 12
Loss: 0.062100159561099565
Training epoch: 13
Loss: 0.06610454915789887
Training epoch: 14
Loss: 0.1039607112204818
Training epoch: 15
Loss: 0.14571654631594277
Training epoch: 16
Loss: 0.059762287500273924
Training epoch: 17
Loss: 0.03109716521398893
Training epoch: 18
Loss: 0.03842195010118936
Training epoch: 19
Loss: 0.014746310347539333
Training epoch: 20
Loss: 0.0060308928365449665
Training epoch: 21
Loss: 0.0015405145764500207
Training epoch: 22
Loss: 0.

In [12]:
# testing the model on val data
correct = 0
total = 0

network.eval()

with torch.no_grad():
  for images, labels in val_loader:

    images = images.to(device)
    labels = labels.to(device)


    outputs = network(images)
    predicted = outputs.argmax(1)

    total += len(labels)
    correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total

print("Accuracy:", accuracy)

Accuracy: 10.733695652173912
